### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [2]:
model_name = "BAAI/bge-large-en-v1.5"
model_kwargs = {"device": "cpu"}

try:
    # Load from disk only if already present
    print(f"Searching for {model_name} locally...")
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name, model_kwargs={**model_kwargs, "local_files_only": True}
    )
    print("Model loaded from local storage.")

except Exception as e:
    # If not found, download it
    print("Model not found locally. Downloading from Hugging Face...")
    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
    print("Download complete and model initialized!")

Searching for BAAI/bge-large-en-v1.5 locally...
Model loaded from local storage.


In [3]:
# Load environment variables
load_dotenv()

# Initialize LLM
llm = ChatGroq(model_name="openai/gpt-oss-20b")  # You can change the model name as needed
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F6B1DE8E60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F6B1D3C1D0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
# Step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [5]:
# Step 2: Vector Store
embedding_model = embeddings
vectorstore = FAISS.from_documents(chunks, embedding_model)

# Step 3:MMR Retriever
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F6B1B187A0>, search_type='mmr', search_kwargs={'k': 5})

In [6]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
"""
)

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F6B1DE8E60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F6B1D3C1D0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [7]:
query_expansion_chain.invoke({"query": "Langchain memory"})

'**Expanded query (Boolean‑style):**  \n\n```\n"LangChain memory" \nOR "LangChain memory module" \nOR "LangChain memory buffer" \nOR "LangChain memory store" \nOR "LangChain memory usage" \nOR "LangChain memory implementation" \nOR "LangChain memory API" \nOR "LangChain memory persistence" \nOR "LangChain memory retrieval" \nOR "LangChain memory architecture" \nOR "LangChain memory examples" \nOR "LangChain memory tutorial" \nOR "LangChain memory best practices" \nOR "LangChain memory system" \nOR "LangChain memory component" \nOR "LangChain memory in chatbots" \nOR "LangChain memory in RAG" \nOR "LangChain memory integration" \nOR "LangChain memory with OpenAI" \nOR "LangChain memory management" \nOR "LLM memory" \nOR "retrieval‑augmented generation memory" \nOR "stateful memory" \nOR "in‑memory store" \nOR "persistent memory" \nOR "vector memory" \nOR "embedding memory"\n```\n\nThis expanded query includes synonyms, technical terms, and contextual phrases that should capture a broade

In [8]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template(
    """
Answer the question based on the context below.

Context:
{context}

Question: {input}
"""
)

document_chain = create_stuff_documents_chain(llm=llm, prompt=answer_prompt)

In [9]:
# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap(
        {
            "input": lambda x: x["input"],
            "context": lambda x: retriever.invoke(
                query_expansion_chain.invoke({"query": x["input"]})
            ),
        }
    )
    | document_chain
)

In [10]:
# Step 6: Runing query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query": query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:  
LangChain memory types, memory implementations, memory modules, memory adapters, memory backends, memory stores, memory buffers, conversation memory, retrieval memory, vector memory, memory architecture, memory usage, memory support, memory capabilities, memory classes, memory categories, LangChain memory support, LangChain memory components, memory management in LangChain, LangChain memory usage, LangChain memory architecture, memory types in LangChain, memory options, memory solutions, memory frameworks, LangChain memory system, LangChain memory API, LangChain memory interface, LangChain memory plugin, LangChain memory strategy, LangChain memory patterns, LangChain memory handling, LangChain memory configuration, LangChain memory integration.
✅ Answer:
 LangChain supports two main types of memory modules that let an LLM keep track of a conversation:

| Memory type | Purpose | How it works |
|-------------|---------|--------------|
| **ConversationBufferMemory** | St

In [11]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query": query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:
```json
{
  "input": "CrewAI agents, CrewAI agent, CrewAI multi‑agent system, CrewAI autonomous agents, CrewAI AI agents, CrewAI software agents, CrewAI agent architecture, CrewAI agent design, CrewAI agent deployment, CrewAI agent behavior, CrewAI agent communication, CrewAI agent coordination, CrewAI agent capabilities, CrewAI agent use cases, CrewAI agent best practices, CrewAI agent integration with LangChain, CrewAI agent integration with LLM, CrewAI agent integration with external APIs, CrewAI agent examples, CrewAI agent tutorials, CrewAI agent documentation, CrewAI agent comparison with other multi‑agent frameworks, CrewAI agent performance, security, scalability, open‑source vs commercial, CrewAI agent training, CrewAI agent evaluation, CrewAI agent monitoring, CrewAI agent debugging, CrewAI agent optimization, CrewAI agent scaling, CrewAI agent lifecycle"
}
```
✅ Answer:
 **CrewAI agents** are the individual LLM‑powered workers that make up a *crew*.  
They ar